
# Final Model Artifacts and Deployment Preparation

## Objectives
- Package the selected weighted ensemble for deployment.
- Preserve the six tuned model pipelines and ensemble weights.
- Save the input feature schema and target information.
- Record final test evaluation metrics.
- Create a reusable prediction function.
- Verify artifact loading and prediction consistency.

The deployed ensemble uses weights selected on the validation set. The held-out test set is reserved for final evaluation and is not used to adjust the model.


In [1]:

from pathlib import Path
import json
import hashlib
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

current_dir = Path.cwd()
possible_roots = [current_dir, current_dir.parent]

PROJECT_ROOT = next(
    (
        path for path in possible_roots
        if (path / "data" / "Processed_527_datapoints.csv").exists()
        and (path / "artifacts" / "feature_config.json").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TUNED_MODEL_DIR = ARTIFACTS_DIR / "tuned_models"
DEPLOYMENT_DIR = ARTIFACTS_DIR / "deployment"
RESULTS_DIR = PROJECT_ROOT / "results"

DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CONFIG_PATH = ARTIFACTS_DIR / "feature_config.json"
WEIGHTS_PATH = RESULTS_DIR / "ensemble_weights.csv"
TEST_RESULTS_PATH = RESULTS_DIR / "final_test_results.csv"
FINAL_SUMMARY_PATH = RESULTS_DIR / "final_evaluation_summary.json"

print("Project root:", PROJECT_ROOT)
print("Deployment directory:", DEPLOYMENT_DIR)

Project root: /Users/divyakandlakuti/Desktop/interpretable-co2-adsorption-ml
Deployment directory: /Users/divyakandlakuti/Desktop/interpretable-co2-adsorption-ml/artifacts/deployment


In [2]:

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

TARGET_COLUMN = feature_config["target"]
FEATURE_COLUMNS = feature_config["features"]

if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise ValueError("Duplicate feature names in feature configuration.")

if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError("The target column must not appear among input features.")

print("Target:", TARGET_COLUMN)
print("Number of features:", len(FEATURE_COLUMNS))
print("Feature order:")
for index, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{index}. {feature}")


Target: CO2 uptake (mmol/g)
Number of features: 10
Feature order:
1. Surface Area (m2/g)
2. Total Pore Volume(cm3/g)
3. Micropore Volume (cm3/g)
4. C (%)
5. H (%)
6. N (%)
7. O (%)
8. S (%)
9. Temp (°C)
10. Pressure (bar)


In [3]:

MODEL_FILES = {
    "Random Forest": "tuned_random_forest.joblib",
    "Extra Trees": "tuned_extra_trees.joblib",
    "Gradient Boosting": "tuned_gradient_boosting.joblib",
    "XGBoost": "tuned_xgboost.joblib",
    "LightGBM": "tuned_lightgbm.joblib",
    "CatBoost": "tuned_catboost.joblib",
}

tuned_pipelines = {}

for model_name, filename in MODEL_FILES.items():
    model_path = TUNED_MODEL_DIR / filename

    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing tuned model: {model_path}. "
            "Run Notebook 06 successfully before continuing."
        )

    tuned_pipelines[model_name] = joblib.load(model_path)

print(f"Loaded {len(tuned_pipelines)} tuned pipelines.")


Loaded 6 tuned pipelines.


In [4]:

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        "Missing ensemble_weights.csv. Run Notebook 07 first."
    )

weights_df = pd.read_csv(WEIGHTS_PATH)

required_columns = {"Model", "Validation_Optimized_Weight"}
if not required_columns.issubset(weights_df.columns):
    raise ValueError("Unexpected columns in the ensemble weights file.")

weight_map = dict(zip(
    weights_df["Model"],
    weights_df["Validation_Optimized_Weight"],
))

if set(weight_map) != set(MODEL_FILES):
    raise ValueError("The weights do not match the six tuned models.")

ensemble_weights = {
    name: float(weight_map[name])
    for name in MODEL_FILES
}

weight_values = np.array(list(ensemble_weights.values()), dtype=float)

if not np.isfinite(weight_values).all():
    raise ValueError("Ensemble weights contain non-finite values.")

if (weight_values < -1e-10).any():
    raise ValueError("Ensemble weights must be non-negative.")

if not np.isclose(weight_values.sum(), 1.0, atol=1e-6):
    raise ValueError("Ensemble weights must sum to one.")

print("Validated ensemble weights:")
display(pd.DataFrame({
    "Model": list(ensemble_weights.keys()),
    "Weight": list(ensemble_weights.values()),
}).round(6))


Validated ensemble weights:


,Model,Weight
0,Random Forest,0.000000
1,Extra Trees,0.000000
2,Gradient Boosting,0.000000
3,XGBoost,0.177019
4,LightGBM,0.276566
5,CatBoost,0.546415


In [5]:

deployment_bundle = {
    "project_name": (
        "Interpretable Ensemble Learning for CO2 Adsorption "
        "on Biomass-Derived Porous Carbons"
    ),
    "target_column": TARGET_COLUMN,
    "feature_columns": FEATURE_COLUMNS,
    "model_order": list(MODEL_FILES.keys()),
    "models": tuned_pipelines,
    "ensemble_weights": ensemble_weights,
    "ensemble_strategy": "Validation-optimized weighted ensemble",
    "prediction_unit": "mmol/g",
    "preprocessing": "Median imputation within each fitted model pipeline",
    "random_state": 42,
}

bundle_path = DEPLOYMENT_DIR / "co2_adsorption_ensemble.joblib"
joblib.dump(deployment_bundle, bundle_path)

print("Saved deployment bundle:", bundle_path.relative_to(PROJECT_ROOT))
print(f"Bundle size: {bundle_path.stat().st_size / (1024 ** 2):.2f} MB")


Saved deployment bundle: artifacts/deployment/co2_adsorption_ensemble.joblib
Bundle size: 13.54 MB


In [6]:

input_schema = {
    "project_name": deployment_bundle["project_name"],
    "target_column": TARGET_COLUMN,
    "target_unit": "mmol/g",
    "features": [
        {
            "name": feature,
            "type": "number",
            "required": True,
        }
        for feature in FEATURE_COLUMNS
    ],
    "missing_value_handling": (
        "The fitted model pipelines apply median imputation to missing "
        "feature values. The API should still validate request structure "
        "and reject non-numeric or non-finite values."
    ),
    "ensemble_strategy": deployment_bundle["ensemble_strategy"],
}

schema_path = DEPLOYMENT_DIR / "input_schema.json"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(input_schema, file, indent=4, ensure_ascii=False)

print("Saved:", schema_path.relative_to(PROJECT_ROOT))


Saved: artifacts/deployment/input_schema.json


In [2]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.pipeline import Pipeline

# ============================================================
# 1. LOCATE THE PROJECT ROOT AUTOMATICALLY
# ============================================================

current_dir = Path.cwd()

possible_roots = [current_dir, *current_dir.parents]

PROJECT_ROOT = next(
    (
        path for path in possible_roots
        if (
            (path / "data" / "Processed_527_datapoints.csv").exists()
            and (path / "artifacts" / "feature_config.json").exists()
            and (path / "artifacts" / "initial_models").is_dir()
            and (path / "artifacts" / "tuned_models").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root.\n"
        f"Current working directory: {current_dir}\n"
        "Expected data/Processed_527_datapoints.csv and "
        "artifacts/feature_config.json."
    )

print("Project root:", PROJECT_ROOT)
print("XGBoost version:", xgb.__version__)


# ============================================================
# 2. DEFINE THE EXISTING AND NEW FILE PATHS
# ============================================================

models = {
    "Initial XGBoost": {
        "original": PROJECT_ROOT
        / "artifacts/initial_models/xgboost.joblib",
        "json": PROJECT_ROOT
        / "artifacts/initial_models/xgboost.json",
        "imputer": PROJECT_ROOT
        / "artifacts/initial_models/xgboost_imputer.joblib",
    },
    "Tuned XGBoost": {
        "original": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost.joblib",
        "json": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost.json",
        "imputer": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost_imputer.joblib",
    },
}

# Check every original artifact before proceeding.
missing_files = [
    str(paths["original"])
    for paths in models.values()
    if not paths["original"].is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following original model files are missing:\n"
        + "\n".join(missing_files)
    )

print("\nBoth original model files were found.")


# ============================================================
# 3. LOAD THE PROJECT FEATURE CONFIGURATION AND DATASET
# ============================================================

config_path = PROJECT_ROOT / "artifacts" / "feature_config.json"
dataset_path = (
    PROJECT_ROOT / "data" / "Processed_527_datapoints.csv"
)

with open(config_path, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

feature_columns = feature_config["features"]

df = pd.read_csv(dataset_path)
df = df.drop_duplicates().reset_index(drop=True)

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Dataset is missing required features: {missing_features}"
    )

# Use the first 20 rows only to verify prediction equivalence.
X_check = df[feature_columns].iloc[:20].copy()

if X_check.empty:
    raise ValueError("The dataset contains no rows to test.")

print("Dataset loaded:", dataset_path.name)
print("Features:", len(feature_columns))
print("Verification rows:", len(X_check))


# ============================================================
# 4. EXPORT EACH MODEL AND VERIFY THE RECONSTRUCTED PIPELINE
# ============================================================

for name, paths in models.items():

    print(f"\n{'=' * 55}")
    print(name)
    print("=" * 55)

    # Load the original fitted scikit-learn pipeline.
    original_pipeline = joblib.load(paths["original"])

    if not hasattr(original_pipeline, "named_steps"):
        raise TypeError(
            f"{name} is not a scikit-learn pipeline."
        )

    steps = original_pipeline.named_steps

    if "model" not in steps or "imputer" not in steps:
        raise ValueError(
            f"Unexpected pipeline structure for {name}. "
            f"Found steps: {list(steps)}"
        )

    original_model = steps["model"]
    fitted_imputer = steps["imputer"]

    if not isinstance(original_model, xgb.XGBRegressor):
        raise TypeError(
            f"Expected XGBRegressor in {name}, "
            f"got {type(original_model)}"
        )

    # Generate reference predictions before exporting.
    original_predictions = np.asarray(
        original_pipeline.predict(X_check)
    )

    # Export the XGBoost model in JSON format.
    original_model.save_model(str(paths["json"]))

    # Preserve the fitted median imputer.
    joblib.dump(fitted_imputer, paths["imputer"])

    print("JSON exported:", paths["json"].relative_to(PROJECT_ROOT))
    print(
        "Imputer saved:",
        paths["imputer"].relative_to(PROJECT_ROOT),
    )

    # Load the JSON into a fresh XGBoost estimator.
    json_model = xgb.XGBRegressor()
    json_model.load_model(str(paths["json"]))

    # Load the saved imputer and rebuild the pipeline.
    json_imputer = joblib.load(paths["imputer"])

    reconstructed_pipeline = Pipeline([
        ("imputer", json_imputer),
        ("model", json_model),
    ])

    # Verify predictions against the original pipeline.
    reconstructed_predictions = np.asarray(
        reconstructed_pipeline.predict(X_check)
    )

    np.testing.assert_allclose(
        original_predictions,
        reconstructed_predictions,
        rtol=1e-5,
        atol=1e-6,
        err_msg=(
            f"{name}: reconstructed predictions do not "
            "match the original predictions."
        ),
    )

    print("JSON model reload: PASSED")
    print("Reconstructed pipeline: PASSED")
    print("Prediction equivalence: PASSED")
    print("Original predictions:", original_predictions[:5])
    print("Reconstructed predictions:", reconstructed_predictions[:5])


# ============================================================
# 5. VERIFY THAT ALL FOUR NEW FILES EXIST
# ============================================================

print("\n" + "=" * 55)
print("EXPORT SUMMARY")
print("=" * 55)

for name, paths in models.items():
    for key in ("json", "imputer"):
        output_path = paths[key]

        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(
                f"Export verification failed: {output_path}"
            )

        print(
            f"{name} | {output_path.relative_to(PROJECT_ROOT)} "
            f"| {output_path.stat().st_size:,} bytes"
        )

print("\nSUCCESS: Both exports passed all Mac-side checks.")
print("Original .joblib files have not been overwritten.")


Project root: /Users/divyakandlakuti/Desktop/interpretable-co2-adsorption-ml
XGBoost version: 3.4.1

Both original model files were found.
Dataset loaded: Processed_527_datapoints.csv
Features: 10
Verification rows: 20

Initial XGBoost
JSON exported: artifacts/initial_models/xgboost.json
Imputer saved: artifacts/initial_models/xgboost_imputer.joblib
JSON model reload: PASSED
Reconstructed pipeline: PASSED
Prediction equivalence: PASSED
Original predictions: [2.514559  2.4312997 2.7593524 2.4872828 3.210449 ]
Reconstructed predictions: [2.514559  2.4312997 2.7593524 2.4872828 3.210449 ]

Tuned XGBoost
JSON exported: artifacts/tuned_models/tuned_xgboost.json
Imputer saved: artifacts/tuned_models/tuned_xgboost_imputer.joblib
JSON model reload: PASSED
Reconstructed pipeline: PASSED
Prediction equivalence: PASSED
Original predictions: [2.5477967 2.8866243 2.8119602 2.4457448 3.219161 ]
Reconstructed predictions: [2.5477967 2.8866243 2.8119602 2.4457448 3.219161 ]

EXPORT SUMMARY
Initial XGB

In [4]:


from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.pipeline import Pipeline

# ============================================================
# 1. LOCATE THE PROJECT ROOT AUTOMATICALLY
# ============================================================

current_dir = Path.cwd()

possible_roots = [current_dir, *current_dir.parents]

PROJECT_ROOT = next(
    (
        path for path in possible_roots
        if (
            (path / "data" / "Processed_527_datapoints.csv").exists()
            and (path / "artifacts" / "feature_config.json").exists()
            and (path / "artifacts" / "initial_models").is_dir()
            and (path / "artifacts" / "tuned_models").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root.\n"
        f"Current working directory: {current_dir}\n"
        "Expected data/Processed_527_datapoints.csv and "
        "artifacts/feature_config.json."
    )

print("Project root:", PROJECT_ROOT)
print("XGBoost version:", xgb.__version__)


# ============================================================
# 2. DEFINE THE EXISTING AND NEW FILE PATHS
# ============================================================

models = {
    "Initial XGBoost": {
        "original": PROJECT_ROOT
        / "artifacts/initial_models/xgboost.joblib",
        "json": PROJECT_ROOT
        / "artifacts/initial_models/xgboost.json",
        "imputer": PROJECT_ROOT
        / "artifacts/initial_models/xgboost_imputer.joblib",
    },
    "Tuned XGBoost": {
        "original": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost.joblib",
        "json": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost.json",
        "imputer": PROJECT_ROOT
        / "artifacts/tuned_models/tuned_xgboost_imputer.joblib",
    },
}

# Check every original artifact before proceeding.
missing_files = [
    str(paths["original"])
    for paths in models.values()
    if not paths["original"].is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following original model files are missing:\n"
        + "\n".join(missing_files)
    )

print("\nBoth original model files were found.")


# ============================================================
# 3. LOAD THE PROJECT FEATURE CONFIGURATION AND DATASET
# ============================================================

config_path = PROJECT_ROOT / "artifacts" / "feature_config.json"
dataset_path = (
    PROJECT_ROOT / "data" / "Processed_527_datapoints.csv"
)

with open(config_path, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

feature_columns = feature_config["features"]

df = pd.read_csv(dataset_path)
df = df.drop_duplicates().reset_index(drop=True)

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Dataset is missing required features: {missing_features}"
    )

# Use the first 20 rows only to verify prediction equivalence.
X_check = df[feature_columns].iloc[:20].copy()

if X_check.empty:
    raise ValueError("The dataset contains no rows to test.")

print("Dataset loaded:", dataset_path.name)
print("Features:", len(feature_columns))
print("Verification rows:", len(X_check))


# ============================================================
# 4. EXPORT EACH MODEL AND VERIFY THE RECONSTRUCTED PIPELINE
# ============================================================

for name, paths in models.items():

    print(f"\n{'=' * 55}")
    print(name)
    print("=" * 55)

    # Load the original fitted scikit-learn pipeline.
    original_pipeline = joblib.load(paths["original"])

    if not hasattr(original_pipeline, "named_steps"):
        raise TypeError(
            f"{name} is not a scikit-learn pipeline."
        )

    steps = original_pipeline.named_steps

    if "model" not in steps or "imputer" not in steps:
        raise ValueError(
            f"Unexpected pipeline structure for {name}. "
            f"Found steps: {list(steps)}"
        )

    original_model = steps["model"]
    fitted_imputer = steps["imputer"]

    if not isinstance(original_model, xgb.XGBRegressor):
        raise TypeError(
            f"Expected XGBRegressor in {name}, "
            f"got {type(original_model)}"
        )

    # Generate reference predictions before exporting.
    original_predictions = np.asarray(
        original_pipeline.predict(X_check)
    )

    # Export the XGBoost model in JSON format.
    original_model.save_model(str(paths["json"]))

    # Preserve the fitted median imputer.
    joblib.dump(fitted_imputer, paths["imputer"])

    print("JSON exported:", paths["json"].relative_to(PROJECT_ROOT))
    print(
        "Imputer saved:",
        paths["imputer"].relative_to(PROJECT_ROOT),
    )

    # Load the JSON into a fresh XGBoost estimator.
    json_model = xgb.XGBRegressor()
    json_model.load_model(str(paths["json"]))

    # Load the saved imputer and rebuild the pipeline.
    json_imputer = joblib.load(paths["imputer"])

    reconstructed_pipeline = Pipeline([
        ("imputer", json_imputer),
        ("model", json_model),
    ])

    # Verify predictions against the original pipeline.
    reconstructed_predictions = np.asarray(
        reconstructed_pipeline.predict(X_check)
    )

    np.testing.assert_allclose(
        original_predictions,
        reconstructed_predictions,
        rtol=1e-5,
        atol=1e-6,
        err_msg=(
            f"{name}: reconstructed predictions do not "
            "match the original predictions."
        ),
    )

    print("JSON model reload: PASSED")
    print("Reconstructed pipeline: PASSED")
    print("Prediction equivalence: PASSED")
    print("Original predictions:", original_predictions[:5])
    print("Reconstructed predictions:", reconstructed_predictions[:5])


# ============================================================
# 5. VERIFY THAT ALL FOUR NEW FILES EXIST
# ============================================================

print("\n" + "=" * 55)
print("EXPORT SUMMARY")
print("=" * 55)

for name, paths in models.items():
    for key in ("json", "imputer"):
        output_path = paths[key]

        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(
                f"Export verification failed: {output_path}"
            )

        print(
            f"{name} | {output_path.relative_to(PROJECT_ROOT)} "
            f"| {output_path.stat().st_size:,} bytes"
        )

print("\nSUCCESS: Both exports passed all Mac-side checks.")
print("Original .joblib files have not been overwritten.")




Project root: /Users/divyakandlakuti/Desktop/interpretable-co2-adsorption-ml
XGBoost version: 3.4.1

Both original model files were found.
Dataset loaded: Processed_527_datapoints.csv
Features: 10
Verification rows: 20

Initial XGBoost
JSON exported: artifacts/initial_models/xgboost.json
Imputer saved: artifacts/initial_models/xgboost_imputer.joblib
JSON model reload: PASSED
Reconstructed pipeline: PASSED
Prediction equivalence: PASSED
Original predictions: [2.514559  2.4312997 2.7593524 2.4872828 3.210449 ]
Reconstructed predictions: [2.514559  2.4312997 2.7593524 2.4872828 3.210449 ]

Tuned XGBoost
JSON exported: artifacts/tuned_models/tuned_xgboost.json
Imputer saved: artifacts/tuned_models/tuned_xgboost_imputer.joblib
JSON model reload: PASSED
Reconstructed pipeline: PASSED
Prediction equivalence: PASSED
Original predictions: [2.5477967 2.8866243 2.8119602 2.4457448 3.219161 ]
Reconstructed predictions: [2.5477967 2.8866243 2.8119602 2.4457448 3.219161 ]

EXPORT SUMMARY
Initial XGB

In [8]:

def predict_co2_uptake(input_data, bundle):
    """
    Predict CO2 uptake for one or more observations.

    Parameters
    ----------
    input_data : pandas.DataFrame or dict
        Input features. Dictionary values may be scalars for a single
        observation or lists for multiple observations.
    bundle : dict
        Loaded deployment bundle.

    Returns
    -------
    numpy.ndarray
        Predicted CO2 uptake in mmol/g.
    """
    feature_columns = bundle["feature_columns"]

    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()
    else:
        raise TypeError("input_data must be a dictionary or DataFrame.")

    missing = [c for c in feature_columns if c not in input_df.columns]
    extra = [c for c in input_df.columns if c not in feature_columns]

    if missing:
        raise ValueError(f"Missing required features: {missing}")

    if extra:
        raise ValueError(f"Unexpected input features: {extra}")

    input_df = input_df[feature_columns].copy()

    for column in feature_columns:
        input_df[column] = pd.to_numeric(input_df[column], errors="raise")

    numeric_values = input_df.to_numpy(dtype=float)

    if np.isinf(numeric_values).any():
        raise ValueError("Input contains infinite values.")

    model_predictions = []

    for model_name in bundle["model_order"]:
        pipeline = bundle["models"][model_name]
        prediction = pipeline.predict(input_df)
        model_predictions.append(prediction)

    prediction_matrix = np.column_stack(model_predictions)

    weights = np.array([
        bundle["ensemble_weights"][name]
        for name in bundle["model_order"]
    ])

    return prediction_matrix @ weights


print("Prediction function defined.")


Prediction function defined.


In [11]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# Locate the project root independently.
current_dir = Path.cwd()
possible_roots = [current_dir, current_dir.parent]

PROJECT_ROOT = next(
    (
        path for path in possible_roots
        if (path / "data" / "Processed_527_datapoints.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/Processed_527_datapoints.csv. "
        "Run this notebook from the project root or notebooks folder."
    )

DATA_PATH = PROJECT_ROOT / "data" / "Processed_527_datapoints.csv"
bundle_path = (
    PROJECT_ROOT / "artifacts" / "deployment"
    / "co2_adsorption_ensemble.joblib"
)

loaded_bundle = joblib.load(bundle_path)

# Create a sample input using the saved feature schema.
sample_df = pd.read_csv(DATA_PATH).drop_duplicates().reset_index(drop=True)
sample_input = sample_df[loaded_bundle["feature_columns"]].iloc[[0]].copy()

bundle_prediction = predict_co2_uptake(sample_input, loaded_bundle)

# Independently calculate the ensemble prediction.
individual_predictions = np.column_stack([
    loaded_bundle["models"][name].predict(sample_input)
    for name in loaded_bundle["model_order"]
])

weights = np.array([
    loaded_bundle["ensemble_weights"][name]
    for name in loaded_bundle["model_order"]
])

expected_prediction = individual_predictions @ weights

print("Bundle prediction:", float(bundle_prediction[0]))
print("Expected prediction:", float(expected_prediction[0]))

assert np.allclose(
    bundle_prediction,
    expected_prediction,
    rtol=1e-6,
    atol=1e-6,
), "Deployment prediction verification failed."

print("Deployment bundle verification passed.")


Bundle prediction: 2.922637893352995
Expected prediction: 2.922637893352995
Deployment bundle verification passed.


In [12]:

import hashlib
import json
import pandas as pd

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


manifest_files = [
    bundle_path,
    schema_path,
    metadata_path,
]

manifest = {
    path.name: {
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }
    for path in manifest_files
}

manifest_path = DEPLOYMENT_DIR / "artifact_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)

print("Checksum manifest saved:", manifest_path)
display(pd.DataFrame([
    {"File": name, **details}
    for name, details in manifest.items()
]))


Checksum manifest saved: /Users/divyakandlakuti/Desktop/interpretable-co2-adsorption-ml/artifacts/deployment/artifact_manifest.json


,File,sha256,size_bytes
0,co2_adsorption_ensemble.joblib,27ed2567dd74ba78ba01ea4226e8ab864b3a441411d472...,14201322
1,input_schema.json,7a990ecc66a569993dc0f79b57c9c050ba59e6fa9ef18a...,1634
2,model_metadata.json,3500bb1f581d4ca71e7090cfe550a3f20bac560427862b...,1192


In [13]:

print("FINAL DEPLOYMENT ARTIFACTS")
print("=" * 45)

for path in [
    bundle_path,
    schema_path,
    metadata_path,
    manifest_path,
]:
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {path.relative_to(PROJECT_ROOT)}")

if not all(
    path.exists()
    for path in [bundle_path, schema_path, metadata_path, manifest_path]
):
    raise FileNotFoundError("One or more deployment artifacts are missing.")

print("\nSelected approach:", deployment_bundle["ensemble_strategy"])
print(f"Test MAE:  {metadata['test_metrics']['MAE']:.4f} mmol/g")
print(f"Test RMSE: {metadata['test_metrics']['RMSE']:.4f} mmol/g")
print(f"Test R²:   {metadata['test_metrics']['R2']:.4f}")

print("\nNotebook 09 completed successfully.")


FINAL DEPLOYMENT ARTIFACTS
[OK] artifacts/deployment/co2_adsorption_ensemble.joblib
[OK] artifacts/deployment/input_schema.json
[OK] artifacts/deployment/model_metadata.json
[OK] artifacts/deployment/artifact_manifest.json

Selected approach: Validation-optimized weighted ensemble
Test MAE:  0.5220 mmol/g
Test RMSE: 0.7064 mmol/g
Test R²:   0.8173

Notebook 09 completed successfully.



---

# Conclusion

The machine learning pipeline for predicting CO₂ uptake in biomass-derived porous carbons has been developed, evaluated, and prepared for deployment.

Six regression models were compared, optimized using Optuna, and evaluated using MAE, RMSE, and R². Ensemble strategies were investigated to determine whether combining predictions could improve performance over individual models.

The selected validation-optimized weighted ensemble achieved the following results on the held-out test set:

- **Mean Absolute Error (MAE):** 0.5220 mmol/g
- **Root Mean Squared Error (RMSE):** 0.7064 mmol/g
- **Coefficient of Determination (R²):** 0.8173

Model explainability was investigated using SHAP, Partial Dependence Plots (PDP), and Accumulated Local Effects (ALE) to better understand the relationships learned by the CatBoost model.

The final deployment artifacts include the trained model pipelines, ensemble weights, input feature schema, model metadata, and artifact integrity manifest. The saved ensemble was verified by checking that its predictions matched those generated independently from the loaded model pipelines.

## Final Outcome

The project establishes an end-to-end machine learning workflow, from dataset preparation and model development to evaluation, interpretability, and deployment preparation. The resulting artifacts are ready to be integrated into the planned FastAPI backend and HTML, CSS, and JavaScript frontend.

## Limitations

- Model performance is constrained by the size, quality, and distribution of the available dataset.
- Test-set results represent performance on this particular held-out sample and do not guarantee equivalent performance on all porous-carbon materials.
- SHAP, PDP, and ALE describe model behavior and do not establish causal relationships.
- Predictions should be interpreted within the feature ranges represented in the training data.
